# 08. Analiza Statystyczna - Bootstrap CI i Testy Statystyczne

## 📋 Cel Notebooka

Ten notebook zawiera analizę statystyczną wyników modelu BERT:

1. **Bootstrap Confidence Intervals (CI)** dla metryk (accuracy, precision, recall, F1)
2. **Testy statystyczne** porównujące różne klasy
3. **Analiza istotności statystycznej** wyników
4. **Table 2: Wyniki modelu z CI** (wymagane dla publikacji)

---

## 🔄 Kolejność Wykonania

Ten notebook powinien być uruchomiony **po** treningu modelu (notebook 05) i ewaluacji (notebook 07).

**Wymagane pliki:**
- `bert_triage_model_*` - wytrenowany model (z notebooka 05) - **automatycznie wykrywany najnowszy model**
- `corpus_val_*.txt` - **TYLKO zbiór walidacyjny** z notebooka 05 (nie cały corpus!) - **automatycznie wykrywany najnowszy plik**

---

## 📊 Metody Statystyczne

### Bootstrap Confidence Intervals
- **Metoda**: Resampling z replacement (n=1000 iteracji)
- **Poziom ufności**: 95% (2.5% i 97.5% percentyle)
- **Metryki**: Accuracy, Precision, Recall, F1-score dla każdej klasy

### Testy Statystyczne
- **McNemar Test**: Porównanie par klas (czy różnice są istotne statystycznie)
- **Chi-square Test**: Test niezależności między klasami
- **Pairwise Comparisons**: Porównanie metryk między klasami

---

## 📈 Table 2: Wyniki Modelu z CI

Table 2 będzie zawierać:
- Klasa
- Precision (z 95% CI)
- Recall (z 95% CI)
- F1-score (z 95% CI)
- Support (liczba próbek)


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
from torch.utils.data import Dataset, DataLoader
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast
from tqdm.auto import tqdm
from pathlib import Path
import glob
import warnings
warnings.filterwarnings('ignore')

# ⚠️ AUTOMATYCZNE WYKRYWANIE NAJNOWSZYCH PLIKÓW
# Znajdź najnowszy model fine-tuned
triage_models = sorted([d for d in glob.glob("bert_triage_model_*") if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Nie znaleziono modelu bert_triage_model_*. Uruchom najpierw notebook 05_Fine_Tune_Model.ipynb")
MODEL_PATH = triage_models[0]  # Najnowszy model
print(f"📂 Znaleziono model: {MODEL_PATH}")

# ⚠️ WAŻNE: Używamy TYLKO zbioru walidacyjnego z notebooka 05 (nie całego corpus!)
val_split_files = sorted(glob.glob("corpus_val_*.txt"), reverse=True)
if not val_split_files:
    raise FileNotFoundError(
        "Nie znaleziono pliku corpus_val_*.txt.\n"
        "Uruchom najpierw notebook 05_Fine_Tune_Model.ipynb, który generuje train/val split."
    )
DATA_FILE = val_split_files[0]  # Najnowszy val split
print(f"📂 Używam zbioru walidacyjnego: {DATA_FILE}")
print("⚠️  UWAGA: To jest TYLKO val split z notebooka 05 (nie cały corpus!)")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
N_BOOTSTRAP = 1000  # Liczba iteracji Bootstrap
CONFIDENCE_LEVEL = 0.95  # 95% CI

# Nazwy klas (wspólny moduł label_mapping — szukamy project root)
import os
import sys
current_dir = Path(os.getcwd())
project_root = None
if (current_dir / "label_mapping.py").exists():
    project_root = current_dir
elif (current_dir / "notebooks_ethos" / "label_mapping.py").exists():
    project_root = current_dir.parent
else:
    for parent in current_dir.parents:
        if (parent / "label_mapping.py").exists():
            project_root = parent
            break
if project_root is None:
    raise FileNotFoundError(
        f"Nie znaleziono pliku label_mapping.py. "
        f"Szukano w: {current_dir} i katalogach nadrzędnych."
    )
sys.path.insert(0, str(project_root))
print(f"📂 Dodano do sys.path: {project_root}")
from label_mapping import TARGET_NAMES, NUM_LABELS
N_CLASSES = NUM_LABELS

# Ustawienia wizualizacji
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

print("📦 Biblioteki załadowane")
print(f"🚀 Device: {DEVICE}")
print(f"📊 Liczba iteracji Bootstrap: {N_BOOTSTRAP}")


## 1. Wczytanie Modelu i Danych Testowych


In [ ]:

class EthosEvalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# ⚠️ WAŻNE: Wczytanie TYLKO zbioru walidacyjnego z notebooka 05 (nie całego corpus!)
# To zapobiega data leakage - dane testowe nie mogą być w zbiorze treningowym
print("📂 Wczytywanie zbioru walidacyjnego...")
texts = []
labels = []

with open(DATA_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()
    
    print(f"   Całkowita liczba linii w val split: {len(lines):,}")
    
    # ⚠️ WAŻNE: NIE shuffle - dane są już podzielone w notebooku 05
    # Używamy WSZYSTKICH linii z val split (to jest już 20% z całego corpus)
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        
        label_str = parts[-1]
        text = " ".join(parts[:-1])
        
        # Mapowanie labeli
        if "POZ" in label_str:
            l_id = 0
        elif "GASTRO" in label_str:
            l_id = 1
        elif "HEMATO" in label_str:
            l_id = 2
        elif "NEFRO" in label_str:
            l_id = 3
        elif "SOR" in label_str:
            l_id = 4
        else:
            continue
        
        texts.append(text)
        labels.append(l_id)

print(f"✅ Zbiór testowy gotowy: {len(texts):,} próbek")

# Rozkład klas
from collections import Counter
label_counts = Counter(labels)
print("\n📊 Rozkład klas w teście:")
for i, name in enumerate(TARGET_NAMES):
    count = label_counts.get(i, 0)
    pct = count / len(labels) * 100 if len(labels) > 0 else 0
    print(f"   {name:15s}: {count:6,} ({pct:5.1f}%)")

# Tokenizer i DataLoader
print(f"\n📥 Ładowanie modelu z: {MODEL_PATH}")
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=N_CLASSES).to(DEVICE)
model.eval()

test_dataset = EthosEvalDataset(texts, labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("✅ Model i dane załadowane")


## 2. Generowanie Predykcji


In [ ]:
# Generowanie predykcji dla całego zbioru testowego
print("🔮 Generowanie predykcji...")

all_probs = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predykcja"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels']
        
        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.array(all_labels)

# Predykcje (argmax)
y_pred = np.argmax(all_probs, axis=1)

print(f"✅ Predykcje wygenerowane: {len(y_pred):,} próbek")

# Podstawowe metryki
accuracy = accuracy_score(all_labels, y_pred)
print("\n📊 Podstawowe metryki:")
print(f"   Accuracy: {accuracy:.4f}")

# Classification report
print("\n📋 Classification Report:")
print(classification_report(all_labels, y_pred, target_names=TARGET_NAMES, digits=4))


## 3. Bootstrap Confidence Intervals

### 3.1. Funkcja Bootstrap


In [ ]:
def bootstrap_metric(y_true, y_pred, metric_func, n_iterations=1000, confidence_level=0.95):
    """
    Oblicza Bootstrap Confidence Interval dla metryki.
    
    Parameters:
    -----------
    y_true : array-like
        Prawdziwe etykiety
    y_pred : array-like
        Predykcje
    metric_func : callable
        Funkcja metryki (np. precision_score, recall_score, f1_score)
    n_iterations : int
        Liczba iteracji Bootstrap
    confidence_level : float
        Poziom ufności (domyślnie 0.95 dla 95% CI)
    
    Returns:
    --------
    metric_value : float
        Wartość metryki na pełnym zbiorze
    ci_lower : float
        Dolna granica CI
    ci_upper : float
        Górna granica CI
    """
    n = len(y_true)
    metric_values = []
    
    # Obliczamy metrykę na pełnym zbiorze
    metric_value = metric_func(y_true, y_pred)
    
    # Bootstrap resampling
    np.random.seed(42)  # Dla reprodukowalności
    for _ in range(n_iterations):
        # Resampling z replacement
        indices = np.random.choice(n, size=n, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        # Obliczamy metrykę na resampled zbiorze
        try:
            boot_metric = metric_func(y_true_boot, y_pred_boot)
            metric_values.append(boot_metric)
        except:
            # Jeśli metryka nie może być obliczona (np. brak klasy w bootstrap), pomijamy
            continue
    
    # Obliczamy percentyle dla CI
    alpha = 1 - confidence_level
    ci_lower = np.percentile(metric_values, 100 * alpha / 2)
    ci_upper = np.percentile(metric_values, 100 * (1 - alpha / 2))
    
    return metric_value, ci_lower, ci_upper

def bootstrap_metric_per_class(y_true, y_pred, metric_func, n_iterations=1000, confidence_level=0.95):
    """
    Oblicza Bootstrap CI dla metryki dla każdej klasy osobno.
    
    Returns:
    --------
    results : dict
        Słownik z wynikami dla każdej klasy: {class_idx: (metric, ci_lower, ci_upper)}
    """
    results = {}
    
    for class_idx in range(N_CLASSES):
        # Binary classification dla każdej klasy
        y_true_binary = (y_true == class_idx).astype(int)
        y_pred_binary = (y_pred == class_idx).astype(int)
        
        # Obliczamy metrykę i CI
        metric_value, ci_lower, ci_upper = bootstrap_metric(
            y_true_binary, y_pred_binary, metric_func, n_iterations, confidence_level
        )
        
        results[class_idx] = (metric_value, ci_lower, ci_upper)
    
    return results

print("✅ Funkcje Bootstrap zdefiniowane")


### 3.2. Obliczanie Bootstrap CI dla wszystkich metryk


In [ ]:
# Obliczanie Bootstrap CI dla wszystkich metryk
print("🔄 Obliczanie Bootstrap CI (to może chwilę potrwać)...")
print(f"   Liczba iteracji: {N_BOOTSTRAP}")
print(f"   Poziom ufności: {CONFIDENCE_LEVEL*100:.0f}%")
print()

# Accuracy (ogólna)
print("📊 Accuracy (ogólna)...")
acc_value, acc_ci_lower, acc_ci_upper = bootstrap_metric(
    all_labels, y_pred, accuracy_score, N_BOOTSTRAP, CONFIDENCE_LEVEL
)
print(f"   Accuracy: {acc_value:.4f} [{acc_ci_lower:.4f}, {acc_ci_upper:.4f}]")

# Precision per class
print("\n📊 Precision (per class)...")
precision_results = bootstrap_metric_per_class(
    all_labels, y_pred, precision_score, N_BOOTSTRAP, CONFIDENCE_LEVEL
)
for class_idx, (prec, ci_l, ci_u) in precision_results.items():
    print(f"   {TARGET_NAMES[class_idx]:15s}: {prec:.4f} [{ci_l:.4f}, {ci_u:.4f}]")

# Recall per class
print("\n📊 Recall (per class)...")
recall_results = bootstrap_metric_per_class(
    all_labels, y_pred, recall_score, N_BOOTSTRAP, CONFIDENCE_LEVEL
)
for class_idx, (rec, ci_l, ci_u) in recall_results.items():
    print(f"   {TARGET_NAMES[class_idx]:15s}: {rec:.4f} [{ci_l:.4f}, {ci_u:.4f}]")

# F1-score per class
print("\n📊 F1-score (per class)...")
f1_results = bootstrap_metric_per_class(
    all_labels, y_pred, f1_score, N_BOOTSTRAP, CONFIDENCE_LEVEL
)
for class_idx, (f1, ci_l, ci_u) in f1_results.items():
    print(f"   {TARGET_NAMES[class_idx]:15s}: {f1:.4f} [{ci_l:.4f}, {ci_u:.4f}]")

print("\n✅ Bootstrap CI obliczone")


## 4. Table 2: Wyniki Modelu z CI

### 4.1. Generowanie Table 2


In [ ]:
# Obliczanie support (liczba próbek dla każdej klasy)
from collections import Counter
support_counts = Counter(all_labels)

# Tworzenie Table 2
table2_data = []

for class_idx in range(N_CLASSES):
    class_name = TARGET_NAMES[class_idx]
    support = support_counts.get(class_idx, 0)
    
    # Precision
    prec, prec_ci_l, prec_ci_u = precision_results[class_idx]
    prec_str = f"{prec:.3f} ({prec_ci_l:.3f}-{prec_ci_u:.3f})"
    
    # Recall
    rec, rec_ci_l, rec_ci_u = recall_results[class_idx]
    rec_str = f"{rec:.3f} ({rec_ci_l:.3f}-{rec_ci_u:.3f})"
    
    # F1-score
    f1, f1_ci_l, f1_ci_u = f1_results[class_idx]
    f1_str = f"{f1:.3f} ({f1_ci_l:.3f}-{f1_ci_u:.3f})"
    
    table2_data.append({
        'Klasa': class_name,
        'Precision (95% CI)': prec_str,
        'Recall (95% CI)': rec_str,
        'F1-score (95% CI)': f1_str,
        'Support': f"{support:,}"
    })

# Dodajemy również ogólną accuracy
table2_data.append({
    'Klasa': 'Overall',
    'Precision (95% CI)': '-',
    'Recall (95% CI)': '-',
    'F1-score (95% CI)': '-',
    'Support': f"{len(all_labels):,}"
})

# Dodajemy accuracy jako osobną linię
table2_data.append({
    'Klasa': 'Accuracy',
    'Precision (95% CI)': f"{acc_value:.3f} ({acc_ci_lower:.3f}-{acc_ci_upper:.3f})",
    'Recall (95% CI)': '-',
    'F1-score (95% CI)': '-',
    'Support': '-'
})

df_table2 = pd.DataFrame(table2_data)

print("📊 Table 2: Wyniki modelu z 95% Confidence Intervals")
print("=" * 80)
display(df_table2)

# Zapis do CSV (dla publikacji)
df_table2.to_csv("table2_model_performance_ci.csv", index=False, encoding='utf-8-sig')
print("\n✅ Table 2 zapisana do: table2_model_performance_ci.csv")


## 5. Testy Statystyczne

### 5.1. McNemar Test (porównanie par klas)


In [ ]:
from scipy.stats import chi2_contingency

def mcnemar_test(y_true, y_pred, class1, class2):
    """
    McNemar test dla porównania dwóch klas.
    Testuje, czy różnice w klasyfikacji między klasami są istotne statystycznie.
    """
    # Tworzymy tabelę kontyngencji dla dwóch klas
    # Tylko przypadki, gdzie prawdziwa klasa to class1 lub class2
    mask = (y_true == class1) | (y_true == class2)
    y_true_binary = y_true[mask]
    y_pred_binary = y_pred[mask]
    
    # Mapowanie: class1 -> 0, class2 -> 1
    y_true_mapped = (y_true_binary == class2).astype(int)
    y_pred_mapped = (y_pred_binary == class2).astype(int)
    
    # Tabela kontyngencji
    #           Pred: class1  Pred: class2
    # True: class1    a         b
    # True: class2    c         d
    a = np.sum((y_true_mapped == 0) & (y_pred_mapped == 0))
    b = np.sum((y_true_mapped == 0) & (y_pred_mapped == 1))
    c = np.sum((y_true_mapped == 1) & (y_pred_mapped == 0))
    d = np.sum((y_true_mapped == 1) & (y_pred_mapped == 1))
    
    contingency_table = np.array([[a, b], [c, d]])
    
    # McNemar test (chi-square test dla tabeli 2x2)
    chi2, p_value = chi2_contingency(contingency_table, correction=True)[:2]
    
    return {
        'class1': TARGET_NAMES[class1],
        'class2': TARGET_NAMES[class2],
        'chi2': chi2,
        'p_value': p_value,
        'significant': p_value < 0.05,
        'contingency_table': contingency_table
    }

# Porównanie wszystkich par klas
print("🔬 McNemar Test - Porównanie par klas")
print("=" * 80)

mcnemar_results = []
for i in range(N_CLASSES):
    for j in range(i + 1, N_CLASSES):
        result = mcnemar_test(all_labels, y_pred, i, j)
        mcnemar_results.append(result)
        
        significance = "✅ Istotne" if result['significant'] else "❌ Nieistotne"
        print(f"\n{TARGET_NAMES[i]} vs {TARGET_NAMES[j]}:")
        print(f"   Chi² = {result['chi2']:.4f}")
        print(f"   p-value = {result['p_value']:.4f}")
        print(f"   {significance} (p < 0.05)")

# Zapis wyników
df_mcnemar = pd.DataFrame([
    {
        'Klasa 1': r['class1'],
        'Klasa 2': r['class2'],
        'Chi²': f"{r['chi2']:.4f}",
        'p-value': f"{r['p_value']:.4f}",
        'Istotne (p<0.05)': 'Tak' if r['significant'] else 'Nie'
    }
    for r in mcnemar_results
])

print("\n📊 Podsumowanie McNemar Test:")
display(df_mcnemar)

df_mcnemar.to_csv("mcnemar_test_results.csv", index=False, encoding='utf-8-sig')
print("\n✅ Wyniki McNemar Test zapisane do: mcnemar_test_results.csv")


### 5.2. Wizualizacja Confidence Intervals


In [ ]:
# Wizualizacja CI dla wszystkich metryk
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_data = {
    'Precision': precision_results,
    'Recall': recall_results,
    'F1-score': f1_results
}

for idx, (metric_name, results) in enumerate(metrics_data.items()):
    ax = axes[idx]
    
    classes = []
    values = []
    ci_lowers = []
    ci_uppers = []
    
    for class_idx in range(N_CLASSES):
        val, ci_l, ci_u = results[class_idx]
        classes.append(TARGET_NAMES[class_idx])
        values.append(val)
        ci_lowers.append(ci_l)
        ci_uppers.append(ci_u)
    
    # Wykres z paskami błędów
    x_pos = np.arange(len(classes))
    ax.errorbar(x_pos, values, yerr=[np.array(values) - np.array(ci_lowers), 
                                     np.array(ci_uppers) - np.array(values)],
                fmt='o', capsize=5, capthick=2, markersize=8, linewidth=2)
    
    ax.set_xlabel('Klasa', fontsize=12)
    ax.set_ylabel(metric_name, fontsize=12)
    ax.set_title(f'{metric_name} z 95% CI', fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Dodajemy wartości na wykresie
    for i, (v, ci_l, ci_u) in enumerate(zip(values, ci_lowers, ci_uppers)):
        ax.text(i, v + 0.05, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('bootstrap_ci_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Wizualizacja zapisana do: bootstrap_ci_visualization.png")
